In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

try:
    df = pd.read_csv('customer_churn.csv')
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'customer_id': [f'CUST_{i:04d}' for i in range(n)],
        'age': np.random.choice([20, 35, 45, 60, np.nan], size=n),
        'tenure_months': np.random.choice([1, 12, 24, 48, np.nan], size=n),
        'monthly_charges': np.random.choice([29.99, 59.99, 89.99, 119.99, np.nan], size=n),
        'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year', None], size=n),
        'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], size=n),
        'tech_support': np.random.choice(['Yes', 'No', None], size=n),
        'churn': np.random.choice([0, 1], size=n, p=[0.73, 0.27])
    })

In [ ]:
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())
print(df['churn'].value_counts())
print(df['churn'].value_counts(normalize=True))

In [ ]:
target_col = 'churn'
X = df.drop(columns=[target_col, 'customer_id'], errors='ignore')
y = df[target_col]

print(target_col)
print(list(X.columns))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
numeric_features = ['age', 'tenure_months', 'monthly_charges']

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [ ]:
categorical_features = ['contract_type', 'payment_method', 'tech_support']

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
], remainder='drop')

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print(X_train.shape, X_train_processed.shape)
print(X_test.shape, X_test_processed.shape)